<a href="https://colab.research.google.com/github/P-franczak/colab-deep-hough-transform/blob/main/Hanqer_Deep_Hough_Transform_Train_SEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorboardX
!pip install POT
!pip install Ninja

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.2/40.2 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 7.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive

# Define a base path in Google Drive for persistent storage
# This will create a 'colab_persistent_data' folder in your Google Drive.
drive_base_path = '/content/drive/MyDrive/colab_persistent_data'

# Mount Google Drive
if not os.path.exists('/content/drive'):
  print("Mounting Google Drive...")
  drive.mount('/content/drive')
else:
  print("Google Drive is already mounted.")

# Ensure the base directory in Drive exists
if not os.path.exists(drive_base_path):
  print(f"Creating base directory '{drive_base_path}' in Google Drive for persistent storage.")
  !mkdir -p "$drive_base_path"

Mounting Google Drive...
Mounted at /content/drive


In [3]:
# Define paths for the repository
repo_name = 'deep-hough-transform'
repo_drive_path = os.path.join(drive_base_path, repo_name)
repo_local_path = repo_name # The directory name after cloning

if os.path.exists(repo_drive_path):
  print(f"Repository '{repo_name}' found in Google Drive. Copying to local environment...")
  !cp -r "$repo_drive_path" .
else:
  print(f"Repository '{repo_name}' not found in Google Drive. Cloning and saving to Drive for future sessions...")
  if not os.path.exists(repo_local_path):
    !git clone https://github.com/P-franczak/deep-hough-transform.git
  # Copy the cloned repository to Google Drive
  !cp -r "$repo_local_path" "$drive_base_path"
  print(f"Repository '{repo_name}' cloned and saved to '{drive_base_path}'.")

Repository 'deep-hough-transform' not found in Google Drive. Cloning and saving to Drive for future sessions...
Cloning into 'deep-hough-transform'...
remote: Enumerating objects: 653, done.
remote: Counting objects: 100% (106/106), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 653 (delta 66), reused 47 (delta 41), pack-reused 547 (from 1)
Receiving objects: 100% (653/653), 13.89 MiB | 34.93 MiB/s, done.
Resolving deltas: 100% (135/135), done.
Repository 'deep-hough-transform' cloned and saved to '/content/drive/MyDrive/colab_persistent_data'.


In [4]:
# Define paths for the SEL dataset zip file
zip_filename = 'ICCV2017_JTLEE_dataset.7z'
zip_drive_path = os.path.join(drive_base_path, zip_filename)
zip_local_path = zip_filename # The filename in the current directory after wget

%cd deep-hough-transform/

if os.path.exists(zip_drive_path):
  print(f"'{zip_filename}' found in Google Drive. Copying to local environment...")
  # Copy to the current directory, which is /content/deep-hough-transform/ after 'cd'

  !cp "$zip_drive_path" .
else:
  print(f"'{zip_filename}' not found in Google Drive. Downloading and saving to Drive for future sessions...")
  !wget https://mcl.korea.ac.kr/research/Submitted/jtlee_slnet/ICCV2017_JTLEE_dataset.7z
  # Copy the downloaded zip file to Google Drive
  !cp "$zip_local_path" "$drive_base_path"
  print(f"'{zip_filename}' downloaded and saved to '{drive_base_path}'.")

/content/deep-hough-transform
'ICCV2017_JTLEE_dataset.7z' found in Google Drive. Copying to local environment...


In [5]:
!7z x ICCV2017_JTLEE_dataset.7z


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan         1 file, 183635100 bytes (176 MiB)

Extracting archive: ICCV2017_JTLEE_dataset.7z
--
Path = ICCV2017_JTLEE_dataset.7z
Type = 7z
Physical Size = 183635100
Headers Size = 30835
Method = LZMA2:25
Solid = +
Blocks = 1

  0%      3% 36 - ICCV2017_JTLEE_images/0037.jpg                                          9% 131 - ICCV2017_JTLEE_images/0132.jpg                                          14% 196 - ICCV2017_JTLEE_images/0197.jpg                                          17% 227 - ICCV

In [6]:
%cd model/_cdht

/content/deep-hough-transform/model/_cdht


In [8]:
!python setup.py build

running build
running build_ext
W0506 13:50:26.781000 6542 torch/utils/cpp_extension.py:535] There are no x86_64-linux-gnu-g++ version bounds defined for CUDA version 12.8
building 'deep_hough' extension
ninja: no work to do.
x86_64-linux-gnu-g++ -fno-strict-overflow -Wsign-compare -DNDEBUG -g -O2 -Wall -g -fstack-protector-strong -Wformat -Werror=format-security -g -fwrapv -O2 -shared -Wl,-O1 -Wl,-Bsymbolic-functions -Wl,-Bsymbolic-functions -g -fwrapv -O2 /content/deep-hough-transform/model/_cdht/build/temp.linux-x86_64-cpython-312/deep_hough_cuda.o /content/deep-hough-transform/model/_cdht/build/temp.linux-x86_64-cpython-312/deep_hough_cuda_kernel.o -L/usr/local/lib/python3.12/dist-packages/torch/lib -L/usr/local/cuda/lib64 -L/usr/lib/x86_64-linux-gnu -lc10 -ltorch -ltorch_cpu -ltorch_python -lcudart -lc10_cuda -ltorch_cuda -o build/lib.linux-x86_64-cpython-312/deep_hough.cpython-312-x86_64-linux-gnu.so


In [9]:
!pip install -e . --user

Obtaining file:///content/deep-hough-transform/model/_cdht
  Preparing metadata (setup.py) ... done
  Running setup.py develop for deep_hough


In [10]:
%cd ../..

/content/deep-hough-transform


In [11]:
!python data/prepare_data_JTLEE.py --root './ICCV2017_JTLEE_images/' --label './ICCV2017_JTLEE_gtlines_all' --save-dir './data/training/JTLEE_resize_100_100/' --list './data/training/JTLEE.lst' --prefix 'JTLEE_resize_100_100' --fixsize 400 --numangle 100 --numrho 100

Processing 0836 [1/1716]...
[ WARN:0@8.552] global loadsave.cpp:1089 imwrite_ Unsupported depth image for selected encoder is fallbacked to CV_8U.
Processing 0412 [2/1716]...
Processing 1302 [3/1716]...
Processing 1408 [4/1716]...
Processing 0594 [5/1716]...
Processing 0026 [6/1716]...
Processing 0432 [7/1716]...
Processing 1104 [8/1716]...
Processing 0060 [9/1716]...
Processing 1486 [10/1716]...
Processing 0986 [11/1716]...
Processing 1185 [12/1716]...
Processing 1581 [13/1716]...
Processing 0238 [14/1716]...
Processing 0451 [15/1716]...
Processing 0953 [16/1716]...
Processing 1321 [17/1716]...
Processing 1404 [18/1716]...
Processing 1495 [19/1716]...
Processing 0861 [20/1716]...
Processing 1112 [21/1716]...
Processing 0705 [22/1716]...
Processing 1585 [23/1716]...
Processing 1040 [24/1716]...
Processing 0470 [25/1716]...
Processing 0788 [26/1716]...
Processing 0658 [27/1716]...
Processing 0715 [28/1716]...
Processing 0833 [29/1716]...
Processing 0546 [30/1716]...
Processing 1463 [31/

In [12]:
!python train.py --resume result/reproduce/checkpoint.pth.tar

2026-05-06 13:53:23,364 INFO {'DATA': {'DIR': 'data/training/', 'VAL_DIR': 'data/training/', 'TEST_DIR': 'data', 'LABEL_FILE': 'data/training/train_1716_100_100.txt', 'VAL_LABEL_FILE': 'data/training/test_1716_100_100.txt', 'TEST_LABEL_FILE': 'data/sel_test.txt', 'BATCH_SIZE': 8, 'WORKERS': 2}, 'OPTIMIZER': {'LR': 0.0002, 'MOMENTUM': 0.9, 'GAMMA': 0.1, 'WEIGHT_DECAY': 0.0, 'STEPS': []}, 'MODEL': {'NUMANGLE': 100, 'NUMRHO': 100, 'FIX': True, 'THRESHOLD': 0.01, 'EDGE_ALIGN': False, 'BACKBONE': 'resnet50'}, 'TRAIN': {'EPOCHS': 5, 'PRINT_FREQ': 100, 'TEST': False, 'SEED': 1997, 'GPU_ID': 0, 'DATA_PARALLEL': False, 'RESUME': None}, 'MISC': {'TMP': './result/reproduce'}}
2026-05-06 13:53:23,364 INFO Namespace(config='./config.yml', resume='result/reproduce/checkpoint.pth.tar', tmp='')
Downloading: "https://download.pytorch.org/models/resnet50-19c8e357.pth" to /root/.cache/torch/hub/checkpoints/resnet50-19c8e357.pth
100% 97.8M/97.8M [00:00<00:00, 388MB/s]
2026-05-06 13:53:25,106 INFO => no ch